# Derivative Pricing | Calculators

-----
##### John Emmanuel Llenarez


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets, Layout, HBox, VBox
from IPython.display import display, clear_output

**Note**: Rerun the cell for each calculator before using it so it will function properly.

## Time/Step Calculator for Binomial Tree

To use this calculator, two variables should be given, else it wont work.

In [56]:
style = {'description_width': '120px'}
layout = widgets.Layout(width='320px')

T_w = widgets.Text(value="1", description='Time (T):', style=style, layout=layout)
N_w = widgets.Text(value="100", description='Steps (N):', style=style, layout=layout)
dt_w = widgets.Text(value='x', description='Time step (dt):', style=style, layout=layout)

calc_button = widgets.Button(
    description='Calculate Parameters',
    button_style='success',
    icon='calculator',
    layout=widgets.Layout(width='320px', margin='10px 0px')
)

res_html = widgets.HTML(value="<div style='color: #666;'>Enter values and click Calculate</div>")

def parse(val):
    v = val.lower().strip()
    try: return float(v)
    except: return 'x'


def on_calculate_clicked(b):
    try:
        T = parse(T_w.value)
        N = parse(N_w.value)
        dt = parse(dt_w.value)
        

        if dt == 'x' and T != 'x' and N != 'x':
            dt = T / N
        elif T == 'x' and dt != 'x' and N != 'x':
            T = dt * N
        elif N == 'x' and dt != 'x' and T != 'x':
            N = int(T / dt)

        if 'x' in [T, N, dt]:
             raise ValueError("Need at least two of (T, N, dt) to solve.")
            
        globals()['T'] = T
        globals()['N'] = N
        globals()['dt'] = dt
        
        res_html.value = f"""
        <div style="border: 2px solid #2e7d32; padding: 15px; background-color: #ffffff; color: #000000; border-radius: 5px;">
            <b style="color: #2e7d32;">Results:</b><hr>
            <p><b>Time (T):</b> {T:.4f}</p>
            <p><b>Steps (N):</b> {N}</p>
            <p><b>dt:</b> {dt:.6f}</p>
        </div>
        """
    except Exception as e:
        res_html.value = f"<div style='color:red; background:#fee; padding:10px;'><b>Error:</b> {str(e)}</div>"

calc_button.on_click(on_calculate_clicked)


ui = widgets.VBox([
    widgets.HTML("<h3>Time Step Calculator</h3>"),
    T_w, N_w, dt_w,
    calc_button,
    res_html
], layout=widgets.Layout(width='350px', padding='10px')) 

display(ui)

## Up/Down Movement Factor Calculator

Now that we can have the dt above, we can use it here to calculate the up or down movement factor for our binomial tree.
Note that there should be at least one given from sigma, u, and d for this calculator to work.

In [57]:
style = {'description_width': '120px'}
layout = widgets.Layout(width='320px')

w = {
    'sig': widgets.Text(value='0.2', description='Sigma (σ):', style=style, layout=layout),
    'u':   widgets.Text(value='x',   description='Up factor (u):', style=style, layout=layout),
    'd':   widgets.Text(value='x',   description='Down factor (d):', style=style, layout=layout),
    'dt':  widgets.Text(value='0.01',description='Time step (dt):', style=style, layout=layout),
}

calc_button = widgets.Button(
    description='Solve for All X',
    button_style='success',
    icon='magic',
    layout=widgets.Layout(width='320px', margin='10px 0px')
)

res_html = widgets.HTML(value="<div style='color: #666;'>Fill in knowns, put 'x' for unknowns.</div>")

def parse_input(val):
    v = val.lower().strip()
    try: 
        return float(v)
    except: 
        return 'x'

def on_calculate_clicked(b):
    try:
        # 1. Access values from the 'w' dictionary
        s_in = parse_input(w['sig'].value)
        u_in = parse_input(w['u'].value)
        d_in = parse_input(w['d'].value)
        dt_in = parse_input(w['dt'].value)

        if not isinstance(dt_in, float):
            raise ValueError("Time step (dt) must be a number.")
        
        dt_val = dt_in

        # 2. The Logic Tree: Solve for u and sigma based on what is available
        # Case A: Sigma is provided
        if isinstance(s_in, float):
            calc_sig = s_in
            calc_u = np.exp(s_in * np.sqrt(dt_val))
            
        # Case B: Sigma is 'x' but Up factor is provided
        elif isinstance(u_in, float):
            calc_u = u_in
            calc_sig = np.log(u_in) / np.sqrt(dt_val)
            
        # Case C: Sigma and Up are 'x' but Down factor is provided
        elif isinstance(d_in, float):
            if d_in <= 0: raise ValueError("Down factor (d) must be positive.")
            calc_u = 1 / d_in
            calc_sig = np.log(calc_u) / np.sqrt(dt_val)
            
        else:
            raise ValueError("Provide Sigma, U, or D as a number.")

        # 3. Finalize d (always 1/u in CRR model)
        calc_d = 1 / calc_u

        # Prepare results for display
        results = {
            'Sigma (σ)': calc_sig,
            'Up factor (u)': calc_u,
            'Down factor (d)': calc_d,
            'Time Step (dt)': dt_val
        }

        # Update globals for notebook use
        globals().update({'u': calc_u, 'd': calc_d, 'dt': dt_val, 'sigma': calc_sig})

        # Generate HTML table rows
        rows = "".join([f"<tr><td style='padding:5px'><b>{k}:</b></td><td>{val:.6f}</td></tr>" for k, val in results.items()])
        
        res_html.value = f"""
        <div style="border: 2px solid #2e7d32; padding: 15px; background-color: #ffffff; color: #000000; border-radius: 5px;">
            <b style="color: #2e7d32;">Master Solver Results:</b><hr>
            <table style="width:100%; color: #000000;">{rows}</table>
        </div>
        """
    except Exception as e:
        res_html.value = f"""
        <div style="background: #331111; color: #ff9999; border: 1px solid #ff4444; padding: 10px; border-radius: 5px;">
            <b>Calculation Error:</b> {str(e)}
        </div>
        """

calc_button.on_click(on_calculate_clicked)

ui = widgets.VBox([
    widgets.HTML("<h3>Up/Down Movement Calculator</h3>"),
    widgets.HTML("<i>Enter 'x' for the variables you want to solve.</i>")
] + list(w.values()) + [calc_button, res_html], 
layout=widgets.Layout(padding='20px', width='380px'))

display(ui)

## Risk-Neutral Probability (p) and Risk-free Interest Rate (r)

We already have the up and down movement factor, which means we can compute for the risk-neutral probability. For this calculator to work, we need to have risk-neutral (p) or risk-free interest rate (r). Note that in binomial, q = (1 - p) and p + q = 1.

In [21]:
style = {'description_width': '120px'}
layout = widgets.Layout(width='320px')

w = {
    'u':   widgets.Text(value='1.1',   description='Up factor (u):', style=style, layout=layout),
    'd':   widgets.Text(value='0.9',   description='Down factor (d):', style=style, layout=layout),
    'dt':  widgets.Text(value='0.01',  description='Time step (dt):', style=style, layout=layout),
    'r':   widgets.Text(value='0.05',  description='Rate (r):', style=style, layout=layout),
    'p':   widgets.Text(value='x',     description='Prob (p):', style=style, layout=layout),
    'q':   widgets.Text(value='x',     description='Prob (q):', style=style, layout=layout),
}

calc_button = widgets.Button(
    description='Solve Probabilities/Rate',
    button_style='success',
    icon='magic',
    layout=widgets.Layout(width='320px', margin='10px 0px')
)

res_html = widgets.HTML(value="<div style='color: #666;'>Enter u, d, dt and ONE of (r, p, or q).</div>")

def parse_input(val):
    v = val.lower().strip()
    try: return float(v)
    except: return 'x'

def on_calculate_clicked(b):
    try:
        # Parse inputs
        u = parse_input(w['u'].value)
        d = parse_input(w['d'].value)
        dt = parse_input(w['dt'].value)
        r_in = parse_input(w['r'].value)
        p_in = parse_input(w['p'].value)
        q_in = parse_input(w['q'].value)

        # Basic validation for u, d, dt
        if any(not isinstance(i, float) for i in [u, d, dt]):
            raise ValueError("u, d, and dt must be numbers.")
        if u <= d:
            raise ValueError("u must be greater than d.")

        # Logic Tree for r, p, and q
        # Scenario 1: Rate (r) is given -> Solve p and q
        if isinstance(r_in, float):
            r_val = r_in
            p_val = (np.exp(r_val * dt) - d) / (u - d)
            q_val = 1 - p_val
        
        # Scenario 2: Probability (p) is given -> Solve r and q
        elif isinstance(p_in, float):
            p_val = p_in
            q_val = 1 - p_val
            # Rearranged formula: r = ln(p(u-d) + d) / dt
            inner = p_val * (u - d) + d
            if inner <= 0: raise ValueError("Invalid p for these u/d values.")
            r_val = np.log(inner) / dt
            
        # Scenario 3: Probability (q) is given -> Solve r and p
        elif isinstance(q_in, float):
            q_val = q_in
            p_val = 1 - q_val
            inner = p_val * (u - d) + d
            if inner <= 0: raise ValueError("Invalid q for these u/d values.")
            r_val = np.log(inner) / dt
            
        else:
            raise ValueError("Provide one of: r, p, or q.")

        results = {
            'Risk-free Rate (r)': r_val,
            'Prob Up (p)': p_val,
            'Prob Down (q)': q_val,
            'Check (p+q)': p_val + q_val
        }
        
        if not (0 < p_val < 1):
            res_html.value += """
            <div style='margin-top:10px; color: #d32f2f; font-size: 0.9em;'>
                ⚠️ <b>Arbitrage Warning:</b> Risk-neutral probability must be between 0 and 1. 
                Check if d < e^(r*dt) < u.
            </div>
            """

        # Update globals
        globals().update({'r': r_val, 'p': p_val, 'q': q_val})

        rows = "".join([f"<tr><td style='padding:5px'><b>{k}:</b></td><td>{val:.6f}</td></tr>" for k, val in results.items()])
        res_html.value = f"""
        <div style="border: 2px solid #2e7d32; padding: 15px; background-color: #ffffff; color: #000000; border-radius: 5px;">
            <b style="color: #2e7d32;">Probability Solver Results:</b><hr>
            <table style="width:100%; color: #000000;">{rows}</table>
        </div>
        """
    except Exception as e:
        res_html.value = f"""
        <div style="background: #331111; color: #ff9999; border: 1px solid #ff4444; padding: 10px; border-radius: 5px;">
            <b>Error:</b> {str(e)}
        </div>
        """

calc_button.on_click(on_calculate_clicked)

ui = widgets.VBox([
    widgets.HTML("<h3>Risk-Neutral Probability & Rate Solver</h3>"),
    widgets.HTML("<i>Set u, d, dt and one variable (r, p, or q) to solve.</i>")
] + list(w.values()) + [calc_button, res_html], 
layout=widgets.Layout(padding='20px', width='380px'))

display(ui)

## Backward Induction of the option payoff
Now we can calculates the value of the option today or the backward induction of the option payoff using
$$C_0 = e^{−rT} [pc_1^u + qc_1^d]$$
Note that, 
- $c_1^u = \max((S_u) - K, 0)$ (Up payoff at time 1)
- $c_1^d = \max((S_d) - K, 0)$ (Down payoff at time 1)
and
- $S_u = S_0 \cdot u$
- $S_d = S_0 \cdot d$

Lets calculate up/down payoff at time 1, then show the value of the option today.

In [26]:
style = {'description_width': '120px'}
layout = widgets.Layout(width='320px')

# Widgets
S_ini_w = widgets.Text(value="100", description='Stock initial (S_ini):', style=style, layout=layout)
K_w = widgets.Text(value="105", description='Strike price (K):', style=style, layout=layout)
u_w = widgets.Text(value='1.2', description='Up factor (u):', style=style, layout=layout)
d_w = widgets.Text(value='0.8', description='Down factor (d):', style=style, layout=layout)

calc_button = widgets.Button(
    description='Calculate Payoffs',
    button_style='success',
    icon='calculator',
    layout=widgets.Layout(width='320px', margin='10px 0px')
)

res_html = widgets.HTML(value="<div style='color: #666;'>Enter values and click Calculate</div>")

def parse(val):
    v = val.lower().strip()
    try: return float(v)
    except: return 'x'

def on_calculate_clicked(b):
    try:
        S_ini = parse(S_ini_w.value)
        K = parse(K_w.value)
        u = parse(u_w.value)
        d = parse(d_w.value) 

        if 'x' in [S_ini, K, u, d]:
            raise ValueError("Please provide numbers for all fields to calculate payoffs.")

        c_u = max((S_ini * u) - K, 0)
        c_d = max((S_ini * d) - K, 0) # Fixed the '=' sign

        res_html.value = f"""
        <div style="border: 2px solid #2e7d32; padding: 15px; background-color: #ffffff; color: #000000; border-radius: 5px;">
            <b style="color: #2e7d32;">1-Step Payoff Results:</b><hr>
            <p><b>Up Payoff (c_u):</b> {c_u:.2f}</p>
            <p><b>Down Payoff (c_d):</b> {c_d:.2f}</p>
            <p style="font-size: 0.9em; color: #666;">Note: These are the values at expiration (Time 1).</p>
        </div>
        """
    except Exception as e:
        res_html.value = f"<div style='color: red;'>Error: {str(e)}</div>"

calc_button.on_click(on_calculate_clicked)

ui = widgets.VBox([
    widgets.HTML("<h3>1-Step Option Payoff Calculator</h3>"),
    S_ini_w, K_w, u_w, d_w,
    calc_button,
    res_html
], layout=widgets.Layout(width='350px', padding='10px')) 

display(ui)

In [32]:
style = {'description_width': '120px'}
layout = widgets.Layout(width='320px')

# Widgets
T_w = widgets.Text(value="1.0", description='Time (T):', style=style, layout=layout)
r_w =  widgets.Text(value='0.05',  description='Rate (r):', style=style, layout=layout)
p_w = widgets.Text(value='0.5',     description='Prob (p):', style=style, layout=layout)
q_w = widgets.Text(value='0.5',     description='Prob (q):', style=style, layout=layout)
c_u_w = widgets.Text(value='20',     description='Up Payoff (c_u):', style=style, layout=layout)
c_d_w = widgets.Text(value='0',     description='Down Payoff (c_d):', style=style, layout=layout)


calc_button = widgets.Button(
    description='Calculate Payoffs',
    button_style='success',
    icon='calculator',
    layout=widgets.Layout(width='320px', margin='10px 0px')
)

res_html = widgets.HTML(value="<div style='color: #666;'>Enter values and click Calculate</div>")

def parse(val):
    v = val.lower().strip()
    try: return float(v)
    except: return 'x'

def on_calculate_clicked(b):
    try:
        T = parse(T_w.value)
        r = parse(r_w.value)
        p = parse(p_w.value)
        q = parse(q_w.value)
        c_u = parse(c_u_w.value)
        c_d = parse(c_d_w.value)

        if 'x' in [T, r, c_u, c_d, p, q]:
            raise ValueError("Please provide numbers for all fields to value of option today.")
            
        c_0 = np.exp(-r * T) * ((p*c_u) + (q*c_d))
        
        res_html.value = f"""
        <div style="border: 2px solid #2e7d32; padding: 15px; background-color: #ffffff; color: #000000; border-radius: 5px;">
            <b style="color: #2e7d32;">Value of option today:</b><hr>
            <p><b>Backward Induction (c_0):</b> {c_0:.2f}</p>
        </div>
        """
    except Exception as e:
        res_html.value = f"<div style='color: red;'>Error: {str(e)}</div>"

calc_button.on_click(on_calculate_clicked)

ui = widgets.VBox([
    widgets.HTML("<h3>Backward Induction Calculator</h3>"),
    r_w, t_w, p_w, q_w, c_u_w, c_d_w,
    calc_button,
    res_html
], layout=widgets.Layout(width='350px', padding='10px')) 

display(ui)

### Delta
The change in the price of the option with respect to the change in the price of the underlying asset.
$$\Delta_0 = \frac{c_1^u − c_1^d}{Su − S_d}$$

In [36]:
style = {'description_width': '120px'}
layout = widgets.Layout(width='320px')

S_ini_w = widgets.Text(value="100", description='Stock initial (S_ini):', style=style, layout=layout)
u_w = widgets.Text(value='1.2', description='Up factor (u):', style=style, layout=layout)
d_w = widgets.Text(value='0.8', description='Down factor (d):', style=style, layout=layout)
c_u_w = widgets.Text(value='20',     description='Up Payoff (c_u):', style=style, layout=layout)
c_d_w = widgets.Text(value='0',     description='Down Payoff (c_d):', style=style, layout=layout)

calc_button = widgets.Button(
    description='Calculate Payoffs',
    button_style='success',
    icon='calculator',
    layout=widgets.Layout(width='320px', margin='10px 0px')
)

res_html = widgets.HTML(value="<div style='color: #666;'>Enter values and click Calculate</div>")

def parse(val):
    v = val.lower().strip()
    try: return float(v)
    except: return 'x'

def on_calculate_clicked(b):
    try:
        S_ini = parse(S_ini_w.value)
        u = parse(u_w.value)
        d = parse(d_w.value)
        c_u = parse(c_u_w.value)
        c_d = parse(c_d_w.value)

        if 'x' in [S_ini, u, d, c_u, c_d]:
            raise ValueError("Please provide numbers for all fields to calculate Delta.")
            
        Delta_0 = (c_u - c_d) / ((S_ini * u) - (S_ini * d))
        
        res_html.value = f"""
        <div style="border: 2px solid #2e7d32; padding: 15px; background-color: #ffffff; color: #000000; border-radius: 5px;">
            <b style="color: #2e7d32;">Delta:</b><hr>
            <p><b>Delta:</b> {Delta_0:.2f}</p>
        </div>
        """
    except Exception as e:
        res_html.value = f"<div style='color: red;'>Error: {str(e)}</div>"

calc_button.on_click(on_calculate_clicked)

ui = widgets.VBox([
    widgets.HTML("<h3>Delta Calculator</h3>"),
    S_ini_w, u_w, d_w, c_u_w, c_d_w,
    calc_button,
    res_html
], layout=widgets.Layout(width='350px', padding='10px')) 

display(ui)

## European Call Option Binomial Tree

If we iterate the process for everything, we can now build a Binomial Tree. 
The calculator will show a binomial tree, but please limit the steps to 20 since more than that wont be readable.

In [47]:
def calculate_and_plot(b):
    S_ini, K, T = s_in.value, k_in.value, t_in.value
    r, u, d, N = r_in.value, u_in.value, d_in.value, n_in.value
    
    dt = T / N 
    p = (np.exp(r * dt) - d) / (u - d) 
    C = np.zeros([N + 1, N + 1]) 
    S = np.zeros([N + 1, N + 1]) 
    Delta = np.zeros([N, N]) 
    
    for i in range(0, N + 1):
        C[N, i] = max(S_ini * (u ** (i)) * (d ** (N - i)) - K, 0)
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
        
    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i])
            S[j, i] = S_ini * (u ** (i)) * (d ** (j - i))
            Delta[j, i] = (C[j + 1, i + 1] - C[j + 1, i]) / (S[j + 1, i + 1] - S[j + 1, i])
            
    with out:
        clear_output(wait=True)
        print(f"--- Option Price: {C[0,0]:.4f} | Risk-Neutral Prob (p): {p:.4f} ---")
        fig, axes = plt.subplots(3, 1, figsize=(15, 20))
        
        for idx, (mat, title) in enumerate([(S, 'Stock'), (C, 'Option'), (Delta, 'Delta')]):
            data = mat 
            mask = np.triu(np.ones_like(data, dtype=bool), k=1)
            sns.heatmap(data, annot=True, fmt=".2f", cmap="YlGnBu", mask=mask, ax=axes[idx], cbar=False)
            axes[idx].set_title(title, fontweight='bold')
            axes[idx].set_xlabel("Number of Up Moves (i)")
            axes[idx].set_ylabel("Time Step (j)")
            
        plt.tight_layout()
        plt.show()

# Widgetts
s_in = widgets.FloatText(value=100.0, description='S_ini:')
k_in = widgets.FloatText(value=100.0, description='K:')
t_in = widgets.FloatText(value=1.0, description='T (Years):')
r_in = widgets.FloatText(value=0.05, description='r:')
u_in = widgets.FloatText(value=1.1, description='u:')
d_in = widgets.FloatText(value=0.9, description='d:')
n_in = widgets.IntText(value=5, description='N Steps:')

calc_btn = widgets.Button(description="Calculate Tree", button_style='success')
calc_btn.on_click(calculate_and_plot)

# Display
out = widgets.Output()
ui = VBox([widgets.Label(value='Binomial Tree for Europian Call Option'),
    HBox([s_in, k_in, t_in]),
    HBox([r_in, u_in, d_in, n_in]),
    calc_btn
])

display(ui, out)

Output()

To read the binomial tree, you will start at the top.
After a time step, the upward movement would be the diagonal, and the downward movement would be just going down from the previous step.

Since we cant show more than 20 steps, the calculator below would be for the calculation of European Price even if N is large.

In [63]:
style = {'description_width': '110px'}
inp_layout = widgets.Layout(width='280px')
box_layout = widgets.Layout(border='1px solid #444', padding='10px', margin='5px', width='320px', border_radius='5px')

S_w = widgets.Text(value="100", description='Stock Price (S):', style=style, layout=inp_layout)
K_w = widgets.Text(value="100", description='Strike (K):', style=style, layout=inp_layout)
T_w = widgets.Text(value="1.0", description='Time (T):', style=style, layout=inp_layout)
r_w = widgets.Text(value='0.05', description='Rate (r):', style=style, layout=inp_layout)
N_w = widgets.Text(value="10", description='Steps (N):', style=style, layout=inp_layout)

u_w = widgets.Text(value='1.1', description='Up factor (u):', style=style, layout=inp_layout)
d_w = widgets.Text(value='0.9', description='Down factor (d):', style=style, layout=inp_layout)
btn_manual = widgets.Button(description='Calc Manual Tree', button_style='success', layout=widgets.Layout(width='280px'))
res_manual = widgets.HTML(value="<div style='color: #666;'>Results will appear here</div>")

sig_w = widgets.Text(value='0.2', description='Volatility (σ):', style=style, layout=inp_layout)
btn_sigma = widgets.Button(description='Calc Sigma Tree', button_style='info', layout=widgets.Layout(width='280px'))
res_sigma = widgets.HTML(value="<div style='color: #666;'>Results will appear here</div>")

def core_calc(S0, K, T, r, N, u, d):
    dt = T / N
    df = np.exp(-r * dt)
    p = (np.exp(r * dt) - d) / (u - d)
    
    if not (d < np.exp(r * dt) < u):
        raise ValueError("Arbitrage! Ensure d < e^(r*dt) < u")

    # Terminal prices
    S_T = S0 * (u ** np.arange(N, -1, -1)) * (d ** np.arange(0, N + 1))
    C = np.maximum(S_T - K, 0)

    for i in range(N - 1, -1, -1):
        if i == 0:
            delta = (C[0] - C[1]) / (S0 * u - S0 * d)
        C = df * (p * C[:-1] + (1 - p) * C[1:])
    return C[0], delta, p

def render_html(price, delta, p, title_color):
    return f"""
    <div style="padding: 10px; background-color: #f9fff9; border-radius: 5px; color: #000; border: 1px solid {title_color};">
        <b style="color: {title_color};">Results:</b><hr>
        <p><b>Price:</b> {price:.4f}</p>
        <p><b>Delta (Δ):</b> {delta:.4f}</p>
        <p><small style="color: #666;">Prob p: {p:.4f}</small></p>
    </div>
    """

def on_manual_click(b):
    try:
        S0, K, T, r, N = float(S_w.value), float(K_w.value), float(T_w.value), float(r_w.value), int(N_w.value)
        u, d = float(u_w.value), float(d_w.value)
        price, delta, p = core_calc(S0, K, T, r, N, u, d)
        res_manual.value = render_html(price, delta, p, "#2e7d32")
    except Exception as e:
        res_manual.value = f"<div style='color:red;'>Error: {str(e)}</div>"

def on_sigma_click(b):
    try:
        S0, K, T, r, N = float(S_w.value), float(K_w.value), float(T_w.value), float(r_w.value), int(N_w.value)
        dt = T / N
        u = np.exp(float(sig_w.value) * np.sqrt(dt))
        d = 1 / u
        price, delta, p = core_calc(S0, K, T, r, N, u, d)
        res_sigma.value = render_html(price, delta, p, "#0277bd")
    except Exception as e:
        res_sigma.value = f"<div style='color:red;'>Error: {str(e)}</div>"

btn_manual.on_click(on_manual_click)
btn_sigma.on_click(on_sigma_click)

header = widgets.HTML("<h3>Binomial Delta Calculator Comparison</h3>")
shared_inputs = widgets.VBox([widgets.HTML("<b>Shared Parameters</b>"),S_w, K_w, T_w, r_w, N_w], layout=widgets.Layout(margin='0 0 20px 0'))

left_side = widgets.VBox([widgets.HTML("<b>Option 1: Manual Factors</b>"), u_w, d_w, btn_manual, res_manual], layout=box_layout)
right_side = widgets.VBox([widgets.HTML("<b>Option 2: Volatility (CRR)</b>"), sig_w, widgets.Label(""), btn_sigma, res_sigma], layout=box_layout)

ui = widgets.VBox([header, shared_inputs, widgets.HBox([left_side, right_side])])
display(ui)

The calculator can now run N<1000 instead of just N<20. But for more than that, we can use the Black-Scholes.
The next calculator will be for European Put, American Call/Put Option Binomial Tree.